# Rearc Data Quest — BLS Productivity Analysis

## Objective

This notebook answers the three analytical questions from the Rearc Data Quest challenge using the curated Gold layer of the BLS Productivity Data Platform.

### Source datasets

- BLS Productivity data
- Data USA annual US population data

### Architecture

Raw Sources → Bronze → Silver → Gold → Challenge Analysis

The challenge-specific logic is intentionally kept outside the production Gold layer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
annual = spark.read.table(
    "bls_dataquest.gold.series_annual_metrics"
)

productivity = spark.read.table(
    "bls_dataquest.gold.fact_productivity"
)

population = spark.read.table(
    "bls_dataquest.gold.population_context"
)

## Question 1 — US Population Statistics

Calculate the mean and standard deviation of annual US population for 2013–2018 inclusive.

Population values are sourced from the current Data USA API dataset.

In [0]:
population_2013_2018 = (
    population
    .filter(
        F.col("year").between(2013, 2018)
    )
    .select(
        "year",
        "population"
    )
    .orderBy("year")
)

display(population_2013_2018)

population_stats = (
    population_2013_2018
    .agg(
        F.count("*").alias("year_count"),
        F.avg("population").alias("mean_population"),
        F.round(F.stddev_pop("population"),0).alias("standard_deviation")
    )
)

display(population_stats)

year,population
2013,311536594
2014,314107084
2015,316515021
2016,318558162
2017,321004407
2018,322903030


year_count,mean_population,standard_deviation
6,3.17437383E8,3886173.0


## Question 2 — Best Year by Series

For each BLS productivity series, determine the year with the largest sum of quarterly `value` observations.

The Gold layer already provides annual aggregation, while the challenge-specific ranking is performed in this notebook.

In [0]:
best_year_window = (
    Window
    .partitionBy("series_id")
    .orderBy(
        F.col("annual_value").desc(),
        F.col("year").desc()
    )
)

best_year_by_series = (
    annual
    .withColumn(
        "rank",
        F.row_number().over(best_year_window)
    )
    .filter(F.col("rank") == 1)
    .select(
        "series_id",
        "series_label",
        "year",
        "annual_value"
    )
    .orderBy("series_id")
)

display(best_year_by_series)

series_id,series_label,year,annual_value
PRS30006011,Manufacturing: All workers: Employment,2022,20.5
PRS30006012,Manufacturing: All workers: Employment,2022,17.099999999999998
PRS30006013,Manufacturing: All workers: Employment,1998,705.895
PRS30006021,Manufacturing: All workers: Average weekly hours,2010,17.7
PRS30006022,Manufacturing: All workers: Average weekly hours,2010,12.399999999999999
PRS30006023,Manufacturing: All workers: Average weekly hours,2014,503.216
PRS30006031,Manufacturing: All workers: Hours worked,2022,20.6
PRS30006032,Manufacturing: All workers: Hours worked,2021,17.0
PRS30006033,Manufacturing: All workers: Hours worked,1998,702.672
PRS30006061,Manufacturing: All workers: Labor compensation,2022,34.5


## Question 3 — Manufacturing Hours Worked

For series `PRS30006032`, return the Q01 observation for each year together with the corresponding US population.

Series:
Manufacturing Sector: Hours Worked for All Workers

In [0]:
manufacturing_q01 = (
    productivity
    .filter(
        (F.col("series_id") == "PRS30006032")
        & (F.col("period") == "Q01")
    )
    .select(
        "year",
        "series_id",
        "series_label",
        "period",
        "value",
        "population"
    )
    .orderBy(F.col("year").desc())
)

display(manufacturing_q01)

year,series_id,series_label,period,value,population
2026,PRS30006032,Manufacturing: All workers: Hours worked,Q01,0.0,null
2025,PRS30006032,Manufacturing: All workers: Hours worked,Q01,-0.6,null
2024,PRS30006032,Manufacturing: All workers: Hours worked,Q01,-0.7,334922499
2023,PRS30006032,Manufacturing: All workers: Hours worked,Q01,0.5,332387540
2022,PRS30006032,Manufacturing: All workers: Hours worked,Q01,5.3,331097593
2021,PRS30006032,Manufacturing: All workers: Hours worked,Q01,0.5,329725481
2020,PRS30006032,Manufacturing: All workers: Hours worked,Q01,-7.0,326569308
2019,PRS30006032,Manufacturing: All workers: Hours worked,Q01,-1.6,324697795
2018,PRS30006032,Manufacturing: All workers: Hours worked,Q01,0.5,322903030
2017,PRS30006032,Manufacturing: All workers: Hours worked,Q01,0.9,321004407
